# Run Arnav's behavioral battery on Qwen (and any VLM) — Drive-persistent

Clones `halli75/Algoverse`, **removes the Gemma-4 model lock** (word-for-word otherwise), **stages EMOTIC**
(reused from Drive if present, cached to Drive if built), runs a chosen SUCCESS experiment with `E2E_MODEL`
set to Qwen / Pixtral / Llama-3.2-Vision / Gemma, and **saves every run to Drive in Arnav's format**
(`results.json` + `heartbeat.json` + `STATUS.md` per run, plus a rolling `SCOREBOARD.md`). Run top-to-bottom.

SUCCESS set: **exp03** cross-modal+dictator · **exp05** ultimatum · **exp06** present-bias ·
**exp09** XSTest over-refusal · **exp10** risk mediation.

**Two ways to run:** the single **▶ One-click** cell just below (install → stage → all models × all
experiments, in one cell), or the step-by-step sections §0–§8 for granular control / debugging.

## ▶ One-click — run the WHOLE sweep in one cell

Self-contained: installs, mounts Drive, clones + patches the battery, stages EMOTIC (Drive-persistent),
and runs **every model × every experiment**, saving + documenting each to Drive. Just run this cell.
Edit the three lists at the top. First run downloads EMOTIC (~1GB) + the models.

In [ ]:
# ------------------------------------------------------------------ edit these 3 lists
SWEEP_MODELS = [
  "google/gemma-4-E4B-it",        # ~4B
  "google/gemma-4-12b-it",        # 12B          (VERIFY exact id)
  "Qwen/Qwen3-VL-2B-Instruct",    # 2B  Qwen3-VL (VERIFY exact id)
  "Qwen/Qwen3-VL-4B-Instruct",    # 4B  Qwen3-VL (VERIFY exact id)
  "Qwen/Qwen3-VL-9B-Instruct",    # 9B  Qwen3-VL (VERIFY exact id)
]
SWEEP_EXPS = ["exp03","exp05","exp06","exp09","exp10"]
SWEEP_TIER = "smoke"              # "full" for final numbers
# ------------------------------------------------------------------ (everything below is automatic)
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","-U","transformers","accelerate","bitsandbytes",
                "gdown","pandas","pillow","scikit-learn","scipy","opencv-python-headless"], check=False)
import os, re, glob, json, base64, hashlib, shutil, pathlib, time
# --- auth ---
try:
    from google.colab import userdata
    for _k in ("HF_TOKEN","XAI_API_KEY"):
        try:
            _v=userdata.get(_k)
            if _v: os.environ[_k]=_v
        except Exception: pass
except Exception: pass
os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", os.environ.get("HF_TOKEN",""))
assert os.environ.get("HF_TOKEN"), ("No HF_TOKEN. In Colab: key icon (left) -> add a secret named HF_TOKEN "
    "(value = your hf_... token) with 'Notebook access' ON, and ACCEPT each model's license on huggingface.co. "
    "Then re-run this cell.")
try:
    from huggingface_hub import login; login(os.environ["HF_TOKEN"])
except Exception as _e: print("hf login note:", _e)
# --- mount Drive ---
from google.colab import drive
try: drive.mount('/content/drive', force_remount=True)
except Exception:
    subprocess.run(["fusermount","-u","/content/drive"], capture_output=True); drive.mount('/content/drive')
DRIVE="/content/drive/MyDrive/affect_refusal"; EMOTIC_DRIVE=f"{DRIVE}/emotic"; RESULTS_DRIVE=f"{DRIVE}/RESULTS/battery_multimodel"
for _d in (EMOTIC_DRIVE, f"{EMOTIC_DRIVE}/emotic_pre", RESULTS_DRIVE): os.makedirs(_d, exist_ok=True)
# --- clone + remove model lock ---
REPO="/content/halli75_algoverse"
if not os.path.isdir(REPO): subprocess.check_call(["git","clone","--depth","1","https://github.com/halli75/Algoverse.git",REPO])
_G="google/gemma-4-E4B-it"
def _patch(path):
    src=open(path,encoding="utf-8").read(); L=src.splitlines(keepends=True); n=0
    for i,ln in enumerate(L):
        if ln.strip()=="if mid != PRIMARY:":
            ind=ln[:len(ln)-len(ln.lstrip())]; L[i]=f"{ind}if False:  # multimodel\n"; n+=1
    out="".join(L)
    out,n2=re.subn(r"env\[(['\"])E2E_MODEL['\"]\]\s*=\s*['\"]"+re.escape(_G)+r"['\"]",
                   lambda m: f"env[{m.group(1)}E2E_MODEL{m.group(1)}] = os.environ.get({m.group(1)}E2E_MODEL{m.group(1)}, {m.group(1)}{_G}{m.group(1)})", out)
    if n+n2 and out!=src: open(path,"w",encoding="utf-8").write(out)
for _p in glob.glob(f"{REPO}/scripts/*.py"): _patch(_p)
# --- stage EMOTIC (Drive-first) ---
EMOTIC_ROOT="/content/emotic_data"; os.makedirs(f"{EMOTIC_ROOT}/emotic_pre", exist_ok=True)
_csv=f"{EMOTIC_ROOT}/emotic_pre/train.csv"; _csvd=f"{EMOTIC_DRIVE}/emotic_pre/train.csv"
def _emo(root):
    for d in pathlib.Path(root).rglob("emotic"):
        if d.is_dir() and next(d.rglob("*.jpg"), None): return d
    return None
if not (os.path.exists(_csv) and os.path.exists(f"{EMOTIC_ROOT}/emotic")):
    _zp=None
    for _c in [f"{EMOTIC_DRIVE}/emotic_images.zip"]+sorted(glob.glob("/content/drive/MyDrive/**/*emotic*.zip", recursive=True), key=lambda p: os.path.getsize(p), reverse=True):
        if os.path.exists(_c) and os.path.getsize(_c)>500_000_000: _zp=_c; break
    if not _zp:
        _zp="/content/emotic_images.zip"; subprocess.check_call(["gdown","https://drive.google.com/uc?id=1icMKzWIlmFKhTkb4OrH8QAHHaaGOP9Zo","-O",_zp,"--fuzzy"])
        try: shutil.copy2(_zp, f"{EMOTIC_DRIVE}/emotic_images.zip")
        except Exception: pass
    if _emo("/content/emotic_images") is None:
        os.makedirs("/content/emotic_images", exist_ok=True); subprocess.check_call(["bash","-lc",f"unzip -qo '{_zp}' -d /content/emotic_images"])
    subprocess.check_call(["ln","-sfn",str(_emo("/content/emotic_images")),f"{EMOTIC_ROOT}/emotic"])
    if os.path.exists(_csvd): shutil.copy2(_csvd,_csv)
    else:
        _az=f"{REPO}/scripts/Annotations.zip"
        if not os.path.exists(_az):
            open("/content/Annotations.zip","wb").write(base64.b64decode(open(f"{REPO}/scripts/Annotations.zip.b64.txt").read().encode())); _az="/content/Annotations.zip"
        subprocess.check_call(["bash","-lc","rm -rf /content/annotations && mkdir -p /content/annotations && unzip -qo "+_az+" -d /content/annotations"])
        subprocess.check_call(["ln","-sfn",str(next(pathlib.Path('/content/annotations').rglob('Annotations'))),f"{EMOTIC_ROOT}/Annotations"])
        _r2="/content/emotic_repo"
        if not os.path.exists(_r2): subprocess.check_call(["git","clone","-q","https://github.com/Tandon-A/emotic.git",_r2])
        subprocess.check_call(["python","mat2py.py","--data_dir",EMOTIC_ROOT,"--label","all"], cwd=_r2)
        try: shutil.copy2(_csv,_csvd)
        except Exception: pass
_njpg=sum(1 for _ in pathlib.Path(f"{EMOTIC_ROOT}/emotic").rglob("*.jpg")); assert _njpg>=20000, f"incomplete emotic {_njpg}"
SPLIT=f"{REPO}/artifacts/colab/emotic_split_1e8ea1c22144dd9d.json"
print("EMOTIC ready:", _njpg, "jpgs")
# --- sweep + document ---
def _persist(EXP, MODEL, TIER, LOG, rd):
    slug=MODEL.replace("/","__"); dst=f"{RESULTS_DRIVE}/{slug}/{EXP}"; os.makedirs(dst, exist_ok=True)
    for f in glob.glob(f"{rd}/*"):
        if os.path.isfile(f):
            try: shutil.copy2(f,dst)
            except Exception: pass
    open(f"{dst}/run.log","w",encoding="utf-8").write(LOG)
    hits=sorted(glob.glob(f"{rd}/results*.json"), key=os.path.getmtime, reverse=True); res=json.load(open(hits[0])) if hits else {}
    open(f"{dst}/STATUS.md","w",encoding="utf-8").write(f"# {EXP} STATUS ({slug})\n- model: {MODEL}\n- tier: {TIER}\n- complete: {res.get('complete')}\n- elapsed_s: {res.get('elapsed_s')}\n- headline: {res.get('headline')}\n- primary: {json.dumps(res.get('primary',{}),default=str)[:1000]}\n")
    board=f"{RESULTS_DRIVE}/SCOREBOARD.md"
    if not os.path.exists(board): open(board,"w",encoding="utf-8").write("# Multimodel battery scoreboard\n\n| ts (UTC) | model | exp | tier | headline | complete | elapsed_s |\n|---|---|---|---|---|---|---|\n")
    open(board,"a",encoding="utf-8").write(f"| {time.strftime('%Y-%m-%d %H:%M', time.gmtime())} | {slug} | {EXP} | {TIER} | {res.get('headline','?')} | {res.get('complete')} | {res.get('elapsed_s')} |\n")
    return res
def _run_one(EXP, MODEL, TIER):
    er="/content/algoverse_run"; rd=f"{er}/battery/{EXP}"; os.makedirs(f"{er}/battery/locks",exist_ok=True); os.makedirs(rd,exist_ok=True)
    for _lp in glob.glob(f"{er}/battery/A100.lock")+glob.glob(f"{er}/battery/locks/*.json"):   # release prior run's GPU lock (runs are sequential)
        try: (shutil.rmtree if os.path.isdir(_lp) else os.remove)(_lp)
        except Exception: pass
    e=dict(os.environ); e.update({"E2E_MODEL":MODEL,"E2E_TIER":TIER,"E2E_ROOT":er,"E2E_EMOTIC":EMOTIC_ROOT,"E2E_SPLIT":SPLIT,
        "E2E_LOCK":f"{er}/battery/A100.lock","PYTHONPATH":f"{REPO}/scripts"+(os.pathsep+os.environ.get('PYTHONPATH','') if os.environ.get('PYTHONPATH') else ''),
        "PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True", "CUDA_VISIBLE_DEVICES":"0"})
    runner=f"{REPO}/scripts/battery_{EXP}_run.py"
    if not os.path.exists(runner): return {"exp":EXP,"model":MODEL.split('/')[-1],"rc":-1,"err":"no runner"}
    t0=time.time(); log=[]
    p=subprocess.Popen([sys.executable,"-u",runner], cwd=REPO, env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout: log.append(line)
    rc=p.wait(); log.append(f"\n=== exit {rc} | wall {time.time()-t0:.0f}s ===\n")
    res=_persist(EXP,MODEL,TIER,"".join(log),rd)
    return {"exp":EXP,"model":MODEL.split('/')[-1],"rc":rc,"headline":res.get("headline"),"complete":res.get("complete")}
_sum=[]
for _M in SWEEP_MODELS:
    for _X in SWEEP_EXPS:
        print(f"[one-click] {_X:6s} on {_M} ...", flush=True)
        try: _r=_run_one(_X,_M,SWEEP_TIER)
        except Exception as _ex: _r={"exp":_X,"model":_M.split('/')[-1],"rc":-1,"err":repr(_ex)[:200]}
        print("   ->", {k:_r.get(k) for k in ("rc","headline","complete","err") if _r.get(k) is not None}, flush=True)
        _sum.append(_r)
        import gc; gc.collect()
        try:
            import torch
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        except Exception: pass
print("\n=== DONE ===")
for _r in _sum: print("  %-26s %-7s rc=%-3s %-26s complete=%s"%(_r.get('model',''),_r.get('exp',''),_r.get('rc',''),str(_r.get('headline') or _r.get('err') or '')[:26],_r.get('complete')))
open(f"{RESULTS_DRIVE}/SWEEP_{time.strftime('%Y%m%d_%H%M', time.gmtime())}.json","w").write(json.dumps(_sum,indent=2,default=str))
print("all results ->", RESULTS_DRIVE)

## 0 · Install

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes gdown pandas pillow scikit-learn scipy opencv-python-headless
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1 · Auth (HF token required for gated models; XAI only for exp03/04 caption-rewrite control)

In [ ]:
import os
try:
    from google.colab import userdata
    for k in ("HF_TOKEN","XAI_API_KEY"):
        try:
            v=userdata.get(k)
            if v: os.environ[k]=v
        except Exception: pass
except Exception: pass
os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", os.environ.get("HF_TOKEN",""))
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")), "| XAI set:", bool(os.environ.get("XAI_API_KEY")))
if not os.environ.get("HF_TOKEN"): print("!! add HF_TOKEN in Colab secrets (needed to download Qwen/Gemma)")

## 2 · Mount Drive + persistent paths

EMOTIC (heavy) and every run's results live under `MyDrive/affect_refusal/` so nothing is re-downloaded or lost.

In [ ]:
from google.colab import drive
try: drive.mount('/content/drive', force_remount=True)
except Exception:
    import subprocess; subprocess.run(["fusermount","-u","/content/drive"], capture_output=True); drive.mount('/content/drive')
DRIVE         = "/content/drive/MyDrive/affect_refusal"
EMOTIC_DRIVE  = f"{DRIVE}/emotic"                 # persistent emotic_images.zip + emotic_pre/train.csv
RESULTS_DRIVE = f"{DRIVE}/RESULTS/battery_multimodel"   # everything compiles under MyDrive/affect_refusal/RESULTS/
for d in (EMOTIC_DRIVE, f"{EMOTIC_DRIVE}/emotic_pre", RESULTS_DRIVE): os.makedirs(d, exist_ok=True)
print("Drive persistent root:", DRIVE)

## 3 · Clone + remove the model lock (inline patch — science untouched)

In [ ]:
import subprocess, re, glob
REPO="/content/halli75_algoverse"
if not os.path.isdir(REPO):
    subprocess.check_call(["git","clone","--depth","1","https://github.com/halli75/Algoverse.git",REPO])
GEMMA="google/gemma-4-E4B-it"
def _patch(path):
    src=open(path,encoding="utf-8").read(); lines=src.splitlines(keepends=True); n=0
    for i,ln in enumerate(lines):
        if ln.strip()=="if mid != PRIMARY:":
            ind=ln[:len(ln)-len(ln.lstrip())]
            lines[i]=f"{ind}if False:  # model-lock removed (multimodel): E2E_MODEL selects the model\n"; n+=1
    out="".join(lines)
    out,n2=re.subn(r"env\[(['\"])E2E_MODEL['\"]\]\s*=\s*['\"]"+re.escape(GEMMA)+r"['\"]",
                   lambda m: f"env[{m.group(1)}E2E_MODEL{m.group(1)}] = os.environ.get({m.group(1)}E2E_MODEL{m.group(1)}, {m.group(1)}{GEMMA}{m.group(1)})", out)
    if n+n2 and out!=src: open(path,"w",encoding="utf-8").write(out)
    return n+n2
print("patched:", sum(_patch(p) for p in glob.glob(f"{REPO}/scripts/*.py")), "edit(s)")
assert not [1 for p in glob.glob(f"{REPO}/scripts/*.py") if "if mid != PRIMARY:" in open(p,encoding="utf-8").read()], "a guard survived"

## 4 · Stage EMOTIC (Drive-first, cache to Drive)

Layout the runners need: `EMOTIC_ROOT/emotic/<Folder>/<Filename>.jpg` + `EMOTIC_ROOT/emotic_pre/train.csv`.
Order: reuse a staged copy → reuse a zip already on Drive → gdown (then cache the zip to Drive). The built
`train.csv` is cached to Drive so later sessions skip the mat2py build.

In [ ]:
import base64, hashlib, shutil, pathlib
EMOTIC_ROOT="/content/emotic_data"; os.makedirs(f"{EMOTIC_ROOT}/emotic_pre", exist_ok=True)
csv_local=f"{EMOTIC_ROOT}/emotic_pre/train.csv"; csv_drive=f"{EMOTIC_DRIVE}/emotic_pre/train.csv"

def _emotic_dir(root):
    for d in pathlib.Path(root).rglob("emotic"):
        if d.is_dir() and next(d.rglob("*.jpg"), None): return d
    return None

if not (os.path.exists(csv_local) and os.path.exists(f"{EMOTIC_ROOT}/emotic")):
    # 1) images: find a usable zip (Drive cache, then any *emotic*.zip on Drive), else gdown
    zp=None
    cands=[f"{EMOTIC_DRIVE}/emotic_images.zip"]+sorted(glob.glob("/content/drive/MyDrive/**/*emotic*.zip", recursive=True), key=lambda p: os.path.getsize(p), reverse=True)
    for c in cands:
        if os.path.exists(c) and os.path.getsize(c)>500_000_000: zp=c; break
    if zp: print("using EMOTIC zip:", zp)
    else:
        zp="/content/emotic_images.zip"
        subprocess.check_call(["gdown","https://drive.google.com/uc?id=1icMKzWIlmFKhTkb4OrH8QAHHaaGOP9Zo","-O",zp,"--fuzzy"])
        try: shutil.copy2(zp, f"{EMOTIC_DRIVE}/emotic_images.zip"); print("cached zip -> Drive")
        except Exception as e: print("zip->Drive cache skipped:", e)
    img_root="/content/emotic_images"
    if _emotic_dir(img_root) is None:
        os.makedirs(img_root, exist_ok=True); subprocess.check_call(["bash","-lc",f"unzip -qo '{zp}' -d {img_root}"])
    em=_emotic_dir(img_root); assert em, "no emotic/ dir with jpgs inside the zip"
    subprocess.check_call(["ln","-sfn",str(em),f"{EMOTIC_ROOT}/emotic"])
    # 2) train.csv: reuse Drive copy, else build via Tandon-A/mat2py from the repo's Annotations.zip
    if os.path.exists(csv_drive):
        shutil.copy2(csv_drive, csv_local); print("reused train.csv from Drive")
    else:
        az=f"{REPO}/scripts/Annotations.zip"
        if not os.path.exists(az):
            open("/content/Annotations.zip","wb").write(base64.b64decode(open(f"{REPO}/scripts/Annotations.zip.b64.txt").read().encode())); az="/content/Annotations.zip"
        subprocess.check_call(["bash","-lc","rm -rf /content/annotations && mkdir -p /content/annotations && unzip -qo "+az+" -d /content/annotations"])
        ann=next(pathlib.Path("/content/annotations").rglob("Annotations"), None); assert ann, "Annotations missing"
        subprocess.check_call(["ln","-sfn",str(ann),f"{EMOTIC_ROOT}/Annotations"])
        repo2="/content/emotic_repo"
        if not os.path.exists(repo2): subprocess.check_call(["git","clone","-q","https://github.com/Tandon-A/emotic.git",repo2])
        m=f"{repo2}/mat2py.py"; t=open(m,encoding="utf-8",errors="ignore").read()
        old=("      cv2.imwrite(os.path.join(save_dir, 'context1.png'), context_arr[-1])\n      cv2.imwrite(os.path.join(save_dir, 'body1.png'), body_arr[-1])")
        new=("      if generate_npy:\n        cv2.imwrite(os.path.join(save_dir, 'context1.png'), context_arr[-1])\n        cv2.imwrite(os.path.join(save_dir, 'body1.png'), body_arr[-1])")
        if old in t: open(m,"w",encoding="utf-8").write(t.replace(old,new))
        subprocess.check_call(["python","mat2py.py","--data_dir",EMOTIC_ROOT,"--label","all"], cwd=repo2)
        try: shutil.copy2(csv_local, csv_drive); print("cached train.csv -> Drive")
        except Exception as e: print("csv->Drive cache skipped:", e)
else:
    print("EMOTIC already staged locally - skip")

n_jpg=sum(1 for _ in pathlib.Path(f"{EMOTIC_ROOT}/emotic").rglob("*.jpg")); print("n_jpg =", n_jpg)
assert n_jpg>=20000, f"incomplete emotic n_jpg={n_jpg}"
SPLIT=f"{REPO}/artifacts/colab/emotic_split_1e8ea1c22144dd9d.json"
sp=json.load(open(SPLIT)); h=hashlib.sha256(json.dumps(sp,sort_keys=True).encode()).hexdigest()[:16]
assert h=="1e8ea1c22144dd9d", "split hash mismatch"; print("split ok", h, "| n_train", len(sp["train_ids"]), "| EMOTIC ->", EMOTIC_ROOT)

## 5 · Config — pick the model + experiment

In [ ]:
EXP   = "exp03"                               # exp03 / exp05 / exp06 / exp09 / exp10
MODEL = "google/gemma-4-E4B-it"               # any of the SWEEP_MODELS below (must be vision-language)
TIER  = "smoke"                               # "smoke" (fast) or "full"

E2E_ROOT="/content/algoverse_run"; os.makedirs(f"{E2E_ROOT}/battery/locks", exist_ok=True); os.makedirs(f"{E2E_ROOT}/battery/{EXP}", exist_ok=True)
import glob as _glob, shutil as _shutil
for _lp in _glob.glob(f"{E2E_ROOT}/battery/A100.lock")+_glob.glob(f"{E2E_ROOT}/battery/locks/*.json"):   # release any stale GPU lock
    try: (_shutil.rmtree if os.path.isdir(_lp) else os.remove)(_lp)
    except Exception: pass
env=dict(os.environ)
env.update({
  "E2E_MODEL": MODEL, "E2E_TIER": TIER,
  "E2E_ROOT": E2E_ROOT, "E2E_EMOTIC": EMOTIC_ROOT, "E2E_SPLIT": SPLIT,
  "E2E_LOCK": f"{E2E_ROOT}/battery/A100.lock",
  "PYTHONPATH": f"{REPO}/scripts" + (os.pathsep+os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else ""),
  "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", "CUDA_VISIBLE_DEVICES": "0",
})
print("will run", EXP, "on", MODEL, "| tier", TIER)
if EXP in ("exp03","exp04") and not env.get("XAI_API_KEY"):
    print("note: no XAI_API_KEY -> exp03/04 caption-rewrite control degrades to weak literal rewrite (rest is fine)")

## 6 · Run (streams + captures the log)

In [ ]:
import sys, time
runner=f"{REPO}/scripts/battery_{EXP}_run.py"; assert os.path.exists(runner), runner
_t0=time.time(); RUN_LOG=[]
p=subprocess.Popen([sys.executable,"-u",runner], cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout: print(line, end=""); RUN_LOG.append(line)
rc=p.wait(); RUN_LOG.append(f"\n=== exit {rc} | wall {time.time()-_t0:.0f}s ===\n"); RUN_LOG="".join(RUN_LOG)
print("\n=== exit", rc, "===")

## 7 · Save to Drive + document (Arnav-style)

Copies the whole run dir (results.json, heartbeat.json, checkpoints, trials, gates) to
`MyDrive/affect_refusal/battery_multimodel/<model>/<exp>/`, writes a `STATUS.md` summary, and appends
a row to the rolling `SCOREBOARD.md`.

In [ ]:
import shutil, glob, time
slug=MODEL.replace("/","__")
dst=f"{RESULTS_DRIVE}/{slug}/{EXP}"; os.makedirs(dst, exist_ok=True)
run_dir=f"{E2E_ROOT}/battery/{EXP}"
for f in glob.glob(f"{run_dir}/*"):
    if os.path.isfile(f):
        try: shutil.copy2(f, dst)
        except Exception as e: print("copy skip", os.path.basename(f), e)
open(f"{dst}/run.log","w",encoding="utf-8").write(RUN_LOG)

hits=sorted(glob.glob(f"{run_dir}/results*.json"), key=os.path.getmtime, reverse=True)
res=json.load(open(hits[0])) if hits else {}
status=(f"# {EXP} STATUS ({slug})\n\n"
        f"- phase: {'done' if res.get('complete') else 'incomplete'}\n"
        f"- model: {MODEL}  (weights {res.get('weights_dtype','?')}, compute {res.get('compute_dtype','?')})\n"
        f"- tier: {TIER}\n- ts: {res.get('updated') or time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}\n"
        f"- elapsed_s: {res.get('elapsed_s')}\n- gpu: {res.get('gpu')}  cuda_visible: {res.get('cuda_visible')}\n"
        f"- n_jpg: {res.get('n_jpg')} | split: {res.get('split_hash')}\n"
        f"- headline: {res.get('headline')}\n- mechanism_answer: {res.get('mechanism_answer')}\n"
        f"- primary: {json.dumps(res.get('primary',{}), default=str)[:1200]}\n"
        f"- gates_summary: {json.dumps(res.get('gates_summary',{}), default=str)[:400]}\n")
open(f"{dst}/STATUS.md","w",encoding="utf-8").write(status)

board=f"{RESULTS_DRIVE}/SCOREBOARD.md"
if not os.path.exists(board):
    open(board,"w",encoding="utf-8").write("# Multimodel battery scoreboard\n\nEvery run appended below. Scores exploratory; do not say photos induce emotion.\n\n| ts (UTC) | model | exp | tier | headline | complete | elapsed_s |\n|---|---|---|---|---|---|---|\n")
open(board,"a",encoding="utf-8").write(
  f"| {time.strftime('%Y-%m-%d %H:%M', time.gmtime())} | {slug} | {EXP} | {TIER} | {res.get('headline','?')} | {res.get('complete')} | {res.get('elapsed_s')} |\n")
print("saved + documented ->", dst)
print("scoreboard ->", board)
print("\n"+status)

## 8 · Sweep — ALL experiments × ALL models at once

Runs every (model, experiment) as its own subprocess (GPU frees between runs), continues past failures, and
saves + documents each run to Drive exactly like §7. Needs §0–§4 done first (does **not** need §5–§7).
Set to `smoke` for a fast full sweep; `full` for final numbers (much longer).

In [ ]:
# All five must be VISION-LANGUAGE (image-prime). NOTE: verify the exact HF ids for the newer
# models below — if one 404s the sweep logs it and moves on (continue-on-error).
SWEEP_MODELS = [
  "google/gemma-4-E4B-it",        # ~4B (Arnav's battery default)
  "google/gemma-4-12b-it",        # 12B          (VERIFY exact id)
  "Qwen/Qwen3-VL-2B-Instruct",    # 2B  Qwen3-VL (VERIFY exact id)
  "Qwen/Qwen3-VL-4B-Instruct",    # 4B  Qwen3-VL (VERIFY exact id)
  "Qwen/Qwen3-VL-9B-Instruct",    # 9B  Qwen3-VL (VERIFY exact id)
]
SWEEP_EXPS  = ["exp03","exp05","exp06","exp09","exp10"]
SWEEP_TIER  = "smoke"          # "full" for final numbers

import sys, time, shutil, glob, gc

def _persist(EXP, MODEL, TIER, RUN_LOG, run_dir):
    slug=MODEL.replace("/","__"); dst=f"{RESULTS_DRIVE}/{slug}/{EXP}"; os.makedirs(dst, exist_ok=True)
    for f in glob.glob(f"{run_dir}/*"):
        if os.path.isfile(f):
            try: shutil.copy2(f, dst)
            except Exception: pass
    open(f"{dst}/run.log","w",encoding="utf-8").write(RUN_LOG)
    hits=sorted(glob.glob(f"{run_dir}/results*.json"), key=os.path.getmtime, reverse=True)
    res=json.load(open(hits[0])) if hits else {}
    open(f"{dst}/STATUS.md","w",encoding="utf-8").write(
        f"# {EXP} STATUS ({slug})\n\n- phase: {'done' if res.get('complete') else 'incomplete'}\n"
        f"- model: {MODEL} (weights {res.get('weights_dtype','?')}, compute {res.get('compute_dtype','?')})\n"
        f"- tier: {TIER}\n- ts: {res.get('updated') or time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}\n"
        f"- elapsed_s: {res.get('elapsed_s')}\n- gpu: {res.get('gpu')}\n- n_jpg: {res.get('n_jpg')} | split: {res.get('split_hash')}\n"
        f"- headline: {res.get('headline')}\n- mechanism_answer: {res.get('mechanism_answer')}\n"
        f"- primary: {json.dumps(res.get('primary',{}), default=str)[:1200]}\n")
    board=f"{RESULTS_DRIVE}/SCOREBOARD.md"
    if not os.path.exists(board):
        open(board,"w",encoding="utf-8").write("# Multimodel battery scoreboard\n\nEvery run appended below. Scores exploratory; do not say photos induce emotion.\n\n| ts (UTC) | model | exp | tier | headline | complete | elapsed_s |\n|---|---|---|---|---|---|---|\n")
    open(board,"a",encoding="utf-8").write(
        f"| {time.strftime('%Y-%m-%d %H:%M', time.gmtime())} | {slug} | {EXP} | {TIER} | {res.get('headline','?')} | {res.get('complete')} | {res.get('elapsed_s')} |\n")
    return res

def run_one(EXP, MODEL, TIER):
    er="/content/algoverse_run"; rd=f"{er}/battery/{EXP}"
    os.makedirs(f"{er}/battery/locks", exist_ok=True); os.makedirs(rd, exist_ok=True)
    for _lp in glob.glob(f"{er}/battery/A100.lock")+glob.glob(f"{er}/battery/locks/*.json"):   # release prior run's GPU lock
        try: (shutil.rmtree if os.path.isdir(_lp) else os.remove)(_lp)
        except Exception: pass
    e=dict(os.environ); e.update({
        "E2E_MODEL":MODEL, "E2E_TIER":TIER, "E2E_ROOT":er, "E2E_EMOTIC":EMOTIC_ROOT, "E2E_SPLIT":SPLIT,
        "E2E_LOCK":f"{er}/battery/A100.lock",
        "PYTHONPATH":f"{REPO}/scripts"+(os.pathsep+os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else ""),
        "PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True", "CUDA_VISIBLE_DEVICES":"0"})
    runner=f"{REPO}/scripts/battery_{EXP}_run.py"
    if not os.path.exists(runner): return {"exp":EXP,"model":MODEL.split('/')[-1],"rc":-1,"err":"no runner"}
    t0=time.time(); log=[]
    p=subprocess.Popen([sys.executable,"-u",runner], cwd=REPO, env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout: log.append(line)
    rc=p.wait(); log.append(f"\n=== exit {rc} | wall {time.time()-t0:.0f}s ===\n")
    res=_persist(EXP, MODEL, TIER, "".join(log), rd)
    return {"exp":EXP,"model":MODEL.split('/')[-1],"rc":rc,"headline":res.get("headline"),"complete":res.get("complete"),"elapsed_s":res.get("elapsed_s")}

summary=[]
for M in SWEEP_MODELS:
    for X in SWEEP_EXPS:
        print(f"[sweep] {X:6s} on {M} ...", flush=True)
        try: r=run_one(X, M, SWEEP_TIER)
        except Exception as ex: r={"exp":X,"model":M.split('/')[-1],"rc":-1,"err":repr(ex)[:200]}
        print("   ->", {k:r.get(k) for k in ("rc","headline","complete","err") if r.get(k) is not None}, flush=True)
        summary.append(r)
        gc.collect()
        try:
            import torch
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        except Exception: pass

print("\n=== sweep summary ===")
print("  %-26s %-7s %-4s %-24s %s"%("model","exp","rc","headline","complete"))
for r in summary:
    print("  %-26s %-7s %-4s %-24s %s"%(r.get("model",""), r.get("exp",""), r.get("rc",""), str(r.get("headline") or r.get("err") or "")[:24], r.get("complete")))
man=f"{RESULTS_DRIVE}/SWEEP_{time.strftime('%Y%m%d_%H%M', time.gmtime())}.json"
open(man,"w",encoding="utf-8").write(json.dumps(summary, indent=2, default=str))
print("\nmanifest ->", man, "| per-run results under", RESULTS_DRIVE)

## 9 · Compile everything into one index

Scans `MyDrive/affect_refusal/RESULTS/` — the black-box battery, the emotion spectrum, and the white-box
mechanism all write there — and writes a single `INDEX.md` listing every result. Run any time.

In [ ]:
import glob, json, os, time
RESULTS_ROOT="/content/drive/MyDrive/affect_refusal/RESULTS"
L=["# Affect study — results index","",f"_compiled {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())}_",""]
for sub,label in [("battery_multimodel","Black-box battery (model x experiment)"),
                  ("emotion_spectrum","Emotion spectrum (VAD)"),
                  ("mechanism","White-box mechanism")]:
    base=f"{RESULTS_ROOT}/{sub}"
    if not os.path.isdir(base): continue
    files=sorted(set(glob.glob(f"{base}/**/*.json", recursive=True)))
    L.append(f"## {label} — `{sub}/`  ({len(files)} file(s))")
    for rf in files:
        try: d=json.load(open(rf))
        except Exception: d={}
        if isinstance(d, list): d={}
        rel=os.path.relpath(rf, RESULTS_ROOT)
        L.append(f"- `{rel}` — model={d.get('model') or d.get('model_id','?')} "
                 f"headline={d.get('headline','')} complete={d.get('complete','')}")
    L.append("")
os.makedirs(RESULTS_ROOT, exist_ok=True)
open(f"{RESULTS_ROOT}/INDEX.md","w",encoding="utf-8").write("\n".join(L))
print("wrote", f"{RESULTS_ROOT}/INDEX.md","\n"); print("\n".join(L))